In [1]:
import os
import glob
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import butter, filtfilt, find_peaks, welch

from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.metrics import mean_absolute_error, r2_score, roc_auc_score

sns.set_theme(style="whitegrid")

RAW_BIDMC_DIR = r"E:\AVIA\data\raw\BIDMC\bidmc_csv"
PROCESSED_BIDMC_DIR = r"E:\AVIA\data\processed\bidmc"
FIGURES_DIR = r"E:\AVIA\reports\figures"
MODELS_DIR = r"E:\AVIA\models"

os.makedirs(PROCESSED_BIDMC_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

signal_files = sorted(glob.glob(os.path.join(RAW_BIDMC_DIR, "bidmc_*_Signals.csv")))
print(f"Found {len(signal_files)} BIDMC Subject Signal CSVs in {RAW_BIDMC_DIR}")

Found 53 BIDMC Subject Signal CSVs in E:\AVIA\data\raw\BIDMC\bidmc_csv


In [2]:
# Inspect Subject 01 to verify header stripping and missingness
s01_sig = pd.read_csv(os.path.join(RAW_BIDMC_DIR, "bidmc_01_Signals.csv"))
s01_num = pd.read_csv(os.path.join(RAW_BIDMC_DIR, "bidmc_01_Numerics.csv"))
s01_br  = pd.read_csv(os.path.join(RAW_BIDMC_DIR, "bidmc_01_Breaths.csv"))

# Fix Bug 1: Strip leading whitespace from column names (' PLETH' -> 'PLETH')
s01_sig.columns = [c.strip() for c in s01_sig.columns]
s01_num.columns = [c.strip() for c in s01_num.columns]
s01_br.columns  = [c.strip() for c in s01_br.columns]

print("Cleaned Signals Columns:", s01_sig.columns.tolist())
print("Cleaned Numerics Columns:", s01_num.columns.tolist())
print("\nMissing Values in Subject 01 Numerics (1 Hz):")
print(s01_num.isnull().sum())
display(s01_num.describe().round(2))

Cleaned Signals Columns: ['Time [s]', 'RESP', 'PLETH', 'V', 'AVR', 'II']
Cleaned Numerics Columns: ['Time [s]', 'HR', 'PULSE', 'RESP', 'SpO2']

Missing Values in Subject 01 Numerics (1 Hz):
Time [s]     0
HR           0
PULSE       13
RESP         0
SpO2        13
dtype: int64


,Time [s],HR,PULSE,RESP,SpO2
count,481.0,481.00,468.00,481.00,468.00
mean,240.0,91.32,91.01,21.44,96.92
std,139.0,1.01,1.14,1.30,0.36
min,0.0,88.00,89.00,20.00,96.00
25%,120.0,91.00,90.00,21.00,97.00
50%,240.0,91.00,91.00,21.00,97.00
75%,360.0,92.00,91.00,22.00,97.00
max,480.0,95.00,94.00,26.00,98.00


In [4]:
FS = 125.0                 # 125 Hz waveform sampling rate
WINDOW_SEC = 30            # 30-second window (3,750 samples) for stable HRV & Resp PSD
STEP_SEC = 10              # 10-second sliding step (~46 windows per subject -> ~2,400 total windows)
WIN_SAMPLES = int(WINDOW_SEC * FS)
STEP_SAMPLES = int(STEP_SEC * FS)

def bandpass_filter(sig, lowcut, highcut, fs=125.0, order=3):
    nyq = 0.5 * fs
    b, a = butter(order, [max(1e-4, lowcut / nyq), min(0.999, highcut / nyq)], btype='band')
    return filtfilt(b, a, np.nan_to_num(sig))

def parse_fix_metadata(fix_path):
    """Extracts Age and Gender from bidmc_XX_Fix.txt."""
    age, sex = 65.0, 0
    if os.path.exists(fix_path):
        with open(fix_path, "r") as f:
            txt = f.read()
        m_age = re.search(r"Age:\s*(\d+)", txt)
        m_sex = re.search(r"Gender:\s*([MF])", txt)
        if m_age: age = float(m_age.group(1))
        if m_sex: sex = 1 if m_sex.group(1) == "M" else 0
    return age, sex

def extract_cardio_resp_window(ppg_w, ecg_w, resp_w, fs=125.0):
    feats = {}
    # 1. PPG (PLETH) Morphology & Pulse-Rate Variability (PRV)
    feats["ppg_mean"] = float(np.mean(ppg_w))
    feats["ppg_std"]  = float(np.std(ppg_w))
    feats["ppg_skew"] = float(pd.Series(ppg_w).skew())
    feats["ppg_kurt"] = float(pd.Series(ppg_w).kurtosis())
    
    ppg_peaks, prop = find_peaks(ppg_w, distance=int(0.30 * fs), prominence=max(1e-4, np.std(ppg_w) * 0.4))
    if len(ppg_peaks) > 2:
        ppi_ms = np.diff(ppg_peaks) / fs * 1000.0
        feats["ppg_pulse_rate"] = float(60000.0 / np.mean(ppi_ms))
        feats["prv_sdnn"]       = float(np.std(ppi_ms))
        feats["prv_rmssd"]      = float(np.sqrt(np.mean(np.square(np.diff(ppi_ms)))))
        feats["ppg_amp_cv"]     = float(np.std(prop["prominences"]) / (np.mean(prop["prominences"]) + 1e-5))
    else:
        feats["ppg_pulse_rate"], feats["prv_sdnn"], feats["prv_rmssd"], feats["ppg_amp_cv"] = 0.0, 0.0, 0.0, 0.0

    # 2. ECG R-Peak Detection & Heart-Rate Variability (HRV)
    ecg_peaks, _ = find_peaks(ecg_w, distance=int(0.30 * fs), prominence=max(1e-4, np.std(ecg_w) * 0.6))
    if len(ecg_peaks) > 2:
        rri_ms = np.diff(ecg_peaks) / fs * 1000.0
        feats["ecg_hr"]    = float(60000.0 / np.mean(rri_ms))
        feats["hrv_sdnn"]  = float(np.std(rri_ms))
        feats["hrv_rmssd"] = float(np.sqrt(np.mean(np.square(np.diff(rri_ms)))))
        feats["hrv_pnn50"] = float(np.mean(np.abs(np.diff(rri_ms)) > 50.0) * 100.0)
    else:
        feats["ecg_hr"], feats["hrv_sdnn"], feats["hrv_rmssd"], feats["hrv_pnn50"] = 0.0, 0.0, 0.0, 0.0

    # 3. Spectral Respiration Modulation from PPG & Resp Impedance (Welch PSD)
    freqs, psd_ppg = welch(ppg_w, fs=fs, nperseg=min(len(ppg_w), 1024))
    _, psd_resp    = welch(resp_w, fs=fs, nperseg=min(len(resp_w), 1024))
    
    resp_band = (freqs >= 0.1) & (freqs <= 0.5)   # 6 to 30 breaths/min
    card_band = (freqs >= 0.6) & (freqs <= 2.5)   # 36 to 150 bpm
    
    feats["ppg_resp_band_power"] = float(np.trapezoid(psd_ppg[resp_band], freqs[resp_band]))
    feats["ppg_card_band_power"] = float(np.trapezoid(psd_ppg[card_band], freqs[card_band]))
    feats["ppg_resp_card_ratio"] = feats["ppg_resp_band_power"] / (feats["ppg_card_band_power"] + 1e-6)
    feats["resp_dom_freq_bpm"]   = float(freqs[resp_band][np.argmax(psd_resp[resp_band])] * 60.0)

    # 4. Direct Impedance Respiration Peaks & Breathing Variability
    r_peaks, _ = find_peaks(resp_w, distance=int(1.5 * fs), prominence=max(1e-4, np.std(resp_w) * 0.3))
    if len(r_peaks) > 1:
        ibi_s = np.diff(r_peaks) / fs
        feats["resp_rate_wave"] = float(60.0 / np.mean(ibi_s))
        feats["resp_ibi_std"]   = float(np.std(ibi_s))
    else:
        feats["resp_rate_wave"], feats["resp_ibi_std"] = feats["resp_dom_freq_bpm"], 0.0

    return feats

all_rows = []
for sig_path in signal_files:
    subj_id = os.path.basename(sig_path).replace("_Signals.csv", "") # e.g., bidmc_01
    num_path = os.path.join(RAW_BIDMC_DIR, f"{subj_id}_Numerics.csv")
    fix_path = os.path.join(RAW_BIDMC_DIR, f"{subj_id}_Fix.txt")
    if not os.path.exists(num_path): continue

    df_s = pd.read_csv(sig_path)
    df_n = pd.read_csv(num_path)
    df_s.columns = [c.strip() for c in df_s.columns]
    df_n.columns = [c.strip() for c in df_n.columns]
    age, sex = parse_fix_metadata(fix_path)

    ecg_col = "II" if "II" in df_s.columns else ("V" if "V" in df_s.columns else df_s.columns[3])
    ppg_f  = bandpass_filter(df_s["PLETH"].values, 0.5, 8.0, FS)
    ecg_f  = bandpass_filter(df_s[ecg_col].values, 0.5, 40.0, FS)
    resp_f = bandpass_filter(df_s["RESP"].values, 0.1, 0.6, FS)

    for start_i in range(0, len(ppg_f) - WIN_SAMPLES + 1, STEP_SAMPLES):
        end_i = start_i + WIN_SAMPLES
        t_start, t_end = start_i / FS, end_i / FS
        
        num_win = df_n[(df_n["Time [s]"] >= t_start) & (df_n["Time [s]"] <= t_end)]
        if num_win.empty: continue

        hr_gt   = num_win["HR"].mean()
        resp_gt = num_win["RESP"].mean()
        spo2_gt = num_win["SpO2"].mean()

        # Data quality check: flag missing/dropout numerics or flatline signals
        quality_ok = int(
            (not np.isnan(hr_gt)) and (not np.isnan(resp_gt)) and (not np.isnan(spo2_gt))
            and (hr_gt > 30) and (resp_gt > 4) and (np.std(ppg_f[start_i:end_i]) > 1e-4)
        )
        if not quality_ok: continue

        row = extract_cardio_resp_window(ppg_f[start_i:end_i], ecg_f[start_i:end_i], resp_f[start_i:end_i], FS)
        row["subject_id"] = subj_id
        row["window_start_sec"] = int(t_start)
        row["age"] = age
        row["sex_male"] = sex
        row["target_hr"] = float(hr_gt)
        row["target_resp"] = float(resp_gt)
        row["target_spo2"] = float(spo2_gt)
        # Prototype cardio-respiratory anomaly indicator (tachypnea >22 or bradypnea <10, hypoxia <94%, tachycardia >105)
        row["cardio_resp_anomaly"] = int((resp_gt > 22) or (resp_gt < 10) or (spo2_gt < 94) or (hr_gt > 105))
        row["data_quality_ok"] = quality_ok
        all_rows.append(row)

df_bidmc = pd.DataFrame(all_rows)
print(f"[+] Extracted {len(df_bidmc)} clean 30s windows across {df_bidmc['subject_id'].nunique()} subjects.")

[+] Extracted 2408 clean 30s windows across 53 subjects.


In [5]:
# Compute Personal Baseline Deviations (using first 5 windows = 90s of each subject as personal reference)
vital_cols = ["ppg_pulse_rate", "prv_rmssd", "ecg_hr", "hrv_rmssd", "hrv_sdnn", "resp_rate_wave", "ppg_resp_card_ratio"]

for subj in df_bidmc["subject_id"].unique():
    mask = df_bidmc["subject_id"] == subj
    base_ref = df_bidmc.loc[mask, vital_cols].head(5).mean()
    for c in vital_cols:
        df_bidmc.loc[mask, f"{c}_personal_delta"] = df_bidmc.loc[mask, c] - base_ref[c]

# Save Master Sanitized + Personal Baseline CSV
master_path = os.path.join(PROCESSED_BIDMC_DIR, "bidmc_features_sanitized_personal_baseline.csv")
df_bidmc.to_csv(master_path, index=False)

# Strict Subject-Level Split (Subjects 01-42 Train, Subjects 43-53 Test -> Zero Subject Leakage!)
unique_subjs = sorted(df_bidmc["subject_id"].unique())
train_subjs = unique_subjs[:42]
test_subjs  = unique_subjs[42:]

train_df = df_bidmc[df_bidmc["subject_id"].isin(train_subjs)].copy()
test_df  = df_bidmc[df_bidmc["subject_id"].isin(test_subjs)].copy()

print(f"Train Set: {train_df.shape} ({len(train_subjs)} subjects) | Test Set: {test_df.shape} ({len(test_subjs)} unseen subjects)")

Train Set: (1902, 34) (42 subjects) | Test Set: (506, 34) (11 unseen subjects)


In [6]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

FEATURE_COLS = [
    "ppg_mean", "ppg_std", "ppg_skew", "ppg_kurt",
    "ppg_pulse_rate", "prv_sdnn", "prv_rmssd", "ppg_amp_cv",
    "ecg_hr", "hrv_sdnn", "hrv_rmssd", "hrv_pnn50",
    "ppg_resp_band_power", "ppg_card_band_power", "ppg_resp_card_ratio",
    "resp_dom_freq_bpm", "resp_rate_wave", "resp_ibi_std"
] + [f"{c}_personal_delta" for c in vital_cols]

# Fit scaler strictly on training subjects only
scaler = RobustScaler()
X_train = scaler.fit_transform(train_df[FEATURE_COLS].values)
X_test  = scaler.transform(test_df[FEATURE_COLS].values)

# 1. Tabular Baseline: Respiratory Rate Regressor & Anomaly Classifier
rr_reg = HistGradientBoostingRegressor(max_iter=150, max_depth=4, learning_rate=0.05, random_state=42)
rr_reg.fit(X_train, train_df["target_resp"].values)
rr_preds = rr_reg.predict(X_test)

anom_clf = HistGradientBoostingClassifier(max_iter=150, max_depth=4, learning_rate=0.05, class_weight="balanced", random_state=42)
anom_clf.fit(X_train, train_df["cardio_resp_anomaly"].values)
anom_probs = anom_clf.predict_proba(X_test)[:, 1]

print(f"[+] Unseen Test Subjects Respiratory Rate MAE: {mean_absolute_error(test_df['target_resp'], rr_preds):.2f} breaths/min")
print(f"[+] Unseen Test Subjects Cardio-Resp Anomaly ROC-AUC: {roc_auc_score(test_df['cardio_resp_anomaly'], anom_probs):.4f}")

# 2. PyTorch AVIACardiacRespEncoder -> 32-Dim Embedding (e_cardiac)
class AVIACardiacRespEncoder(nn.Module):
    def __init__(self, input_dim: int, embed_dim: int = 32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(64, embed_dim),
            nn.BatchNorm1d(embed_dim),
            nn.SiLU()
        )
        self.anom_head = nn.Linear(embed_dim, 1)
        self.rr_head   = nn.Linear(embed_dim, 1)

    def forward(self, x):
        emb = self.encoder(x)
        anom_prob = torch.sigmoid(self.anom_head(emb)).squeeze(-1)
        rr_est    = self.rr_head(emb).squeeze(-1)
        return anom_prob, rr_est, emb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
model = AVIACardiacRespEncoder(len(FEATURE_COLS), 32).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
bce_loss = nn.BCELoss()
l1_loss  = nn.SmoothL1Loss()

train_ds = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(train_df["cardio_resp_anomaly"].values, dtype=torch.float32),
    torch.tensor(train_df["target_resp"].values, dtype=torch.float32)
)
loader = DataLoader(train_ds, batch_size=64, shuffle=True)

model.train()
for epoch in range(25):
    for bx, by_anom, by_rr in loader:
        bx, by_anom, by_rr = bx.to(device), by_anom.to(device), by_rr.to(device)
        optimizer.zero_grad()
        p_anom, p_rr, _ = model(bx)
        loss = bce_loss(p_anom, by_anom) + 0.1 * l1_loss(p_rr, by_rr)
        loss.backward()
        optimizer.step()

# Export 32-Dim Embeddings for All BIDMC Windows
model.eval()
X_all_scaled = scaler.transform(df_bidmc[FEATURE_COLS].values)
with torch.no_grad():
    p_anom_all, p_rr_all, emb_all = model(torch.tensor(X_all_scaled, dtype=torch.float32, device=device))
    emb_np = emb_all.cpu().numpy()

torch.save(model.state_dict(), os.path.join(MODELS_DIR, "avia_cardiac_resp_encoder.pt"))
emb_df = pd.DataFrame(emb_np, columns=[f"cardiac_emb_{i}" for i in range(32)])
emb_df.insert(0, "subject_id", df_bidmc["subject_id"])
emb_df.insert(1, "cardio_resp_anomaly_prob", np.round(p_anom_all.cpu().numpy(), 4))
emb_df.insert(2, "est_resp_rate", np.round(p_rr_all.cpu().numpy(), 2))
emb_df.insert(3, "data_quality_ok", df_bidmc["data_quality_ok"])

emb_path = os.path.join(PROCESSED_BIDMC_DIR, "bidmc_32dim_embeddings.csv")
emb_df.to_csv(emb_path, index=False)
print(f"[+] Saved 32-Dim Cardiac-Respiratory Embeddings -> {emb_path} (Shape: {emb_df.shape})")

[+] Unseen Test Subjects Respiratory Rate MAE: 0.82 breaths/min
[+] Unseen Test Subjects Cardio-Resp Anomaly ROC-AUC: 0.8324
[+] Saved 32-Dim Cardiac-Respiratory Embeddings -> E:\AVIA\data\processed\bidmc\bidmc_32dim_embeddings.csv (Shape: (2408, 36))


In [7]:
df_bidmc=pd.read_csv(r"E:\AVIA\data\processed\bidmc\bidmc_32dim_embeddings.csv")
df_bidmc.head(10)

,subject_id,cardio_resp_anomaly_prob,est_resp_rate,data_quality_ok,cardiac_emb_0,cardiac_emb_1,cardiac_emb_2,cardiac_emb_3,cardiac_emb_4,cardiac_emb_5,...,cardiac_emb_22,cardiac_emb_23,cardiac_emb_24,cardiac_emb_25,cardiac_emb_26,cardiac_emb_27,cardiac_emb_28,cardiac_emb_29,cardiac_emb_30,cardiac_emb_31
0,bidmc_01,0.2764,22.09,1,1.905044,1.336473,0.356510,1.013370,1.869807,1.413200,...,-0.224738,-0.275290,-0.246135,0.115095,-0.230106,2.041980,1.376659,0.536344,0.109082,1.857577
1,bidmc_01,0.1419,20.97,1,2.190544,1.102848,0.376988,0.744067,1.149622,1.201895,...,-0.200370,-0.273232,-0.190051,0.141376,-0.235752,1.675031,1.455863,0.390462,0.192940,1.323740
2,bidmc_01,0.4512,21.81,1,2.032377,2.116147,0.161019,0.868594,1.117129,1.142744,...,-0.261065,-0.259262,-0.251716,0.284507,-0.175299,2.410150,1.640145,0.313327,0.095390,1.726569
3,bidmc_01,0.2548,20.72,1,2.207005,2.413076,0.223551,0.579498,0.603790,0.930599,...,-0.271925,-0.272845,-0.254431,0.530827,-0.109369,2.189347,2.021380,0.235349,0.502610,1.150080
4,bidmc_01,0.3634,20.61,1,2.205167,2.678846,0.208816,0.647899,0.535877,0.914793,...,-0.277790,-0.260638,-0.273919,0.656362,-0.041807,2.447437,2.043474,0.281069,0.655687,1.212534
5,bidmc_01,0.1067,19.95,1,2.108509,2.525326,0.201915,0.793779,0.537057,0.536950,...,-0.277392,-0.272624,-0.236088,0.678047,-0.056682,1.824581,2.255129,0.715807,1.031688,0.590441
6,bidmc_01,0.1068,20.26,1,2.094078,2.010188,-0.027918,0.845021,0.705225,0.571794,...,-0.265337,-0.277300,-0.166227,0.484471,-0.125394,1.735907,2.157898,0.768353,0.447972,0.699959
7,bidmc_01,0.5124,18.85,1,1.783420,3.089663,-0.107829,-0.230339,-0.192493,-0.276325,...,-0.274752,1.121017,-0.118185,-0.160927,0.117621,1.842716,3.446814,1.339666,-0.221718,0.146465
8,bidmc_01,0.4407,17.03,1,1.854123,2.930855,-0.230161,-0.273854,0.183269,-0.066594,...,-0.265865,-0.018613,-0.136866,0.114382,0.163816,1.682643,2.600752,0.605185,-0.205534,0.266644
9,bidmc_01,0.9747,21.34,1,2.085445,3.069513,1.713671,-0.195304,1.212250,1.870223,...,-0.073624,0.617314,-0.090419,0.550342,-0.175437,3.488547,-0.041091,-0.019051,0.357299,3.256664
